In [10]:
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as spln

import matplotlib
matplotlib.use("QtAgg")  # Forces interactive desktop rendering window
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [12]:
class QuantumSimulation:
    """Simulates a 1D quantum wave packet interacting with a double potential barrier."""
    
    def __init__(self, n_points=800, dt=0.4, x_bounds=(-200.0, 200.0), 
                 sigma0=6.0, k0=2.0, x0=-150.0, 
                 barrier_height=1.5, barrier_width=15.0, central_barrier_width=6.0):
        
        self.n_points = n_points
        self.dt = dt
        
        # 1. Grid Discretization (Fixed tuple unpacking)
        self.x, self.dx = np.linspace(x_bounds[0], x_bounds[1], n_points, retstep=True)
        
        # 2. Double Barrier Potential Setup (Positive height makes walls point UP)
        half_gap = central_barrier_width / 2.0
        left_wall_start = -half_gap - barrier_width
        right_wall_end = half_gap + barrier_width
        
        self.potential = np.where(
            ((self.x >= left_wall_start) & (self.x <= -half_gap)) |
            ((self.x >= half_gap) & (self.x <= right_wall_end)),
            barrier_height,
            0.0
        )
        
        # 3. Wave Function Initialization (Normalized Gaussian)
        norm_factor = (2.0 * np.pi * sigma0**2)**(-0.25)
        self.psi = norm_factor * np.exp(-(self.x - x0)**2 / (4.0 * sigma0**2)) * np.exp(1.0j * k0 * self.x)
        self.prob_density = np.abs(self.psi)**2
        
        # 4. Complex Absorbing Boundary Mask (Fixed array-comparison typos)
        self.mask = np.ones_like(self.x)
        L = 30
        left = self.x < self.x[0] + L  
        right = self.x > self.x[-1] - L

        self.mask[left] = np.sin(np.pi * (self.x[left] - self.x[0]) / (2 * L))**0.15
        self.mask[right] = np.sin(np.pi * (self.x[-1] - self.x[right]) / (2 * L))**0.15
        
        # 5. Hamiltonian Matrix Construction
        diag = (1.0 / self.dx**2) + self.potential
        off_diag = np.full(n_points - 1, -0.5 / self.dx**2)
        hamiltonian = sparse.diags([diag, off_diag, off_diag], [0, 1, -1], format='csc')
        
        # 6. Crank-Nicolson Operator Setup
        identity = sparse.eye(n_points, format='csc')
        implicit_matrix = identity - (dt / 2.0j) * hamiltonian
        self.explicit_matrix = identity + (dt / 2.0j) * hamiltonian
        
        # Pre-factorize for numerical stability and high speeds
        self.solver = spln.factorized(implicit_matrix)

    def step(self):
        """Advances the wave function by dt and applies absorbing masks."""
        rhs = self.explicit_matrix @ self.psi
        self.psi = self.solver(rhs)
        
        # Apply boundary absorption to stop edge reflections
        self.psi *= self.mask
        
        # Re-normalization
        self.prob_density = np.abs(self.psi)**2
        norm = np.sqrt(np.sum(self.prob_density) * self.dx)
        self.psi /= norm
        self.prob_density = np.abs(self.psi)**2
        
        return self.prob_density

    def transmission(self):
        mask = self.x > 20
        return np.sum(self.prob_density[mask]) * self.dx

    def reflection(self):
        mask = self.x < -20
        return np.sum(self.prob_density[mask]) * self.dx

    def expectation_x(self):
        return np.sum(self.x * self.prob_density) * self.dx


class QuantumAnimator:
    """Handles rendering layout, canvas element bindings, and live dashboard text updates."""
    
    def __init__(self, simulation):
        self.sim = simulation
        self.time = 0.0
        
        # Setup Plot Window Layout
        self.fig, self.ax = plt.subplots(figsize=(10, 6))
        self.ax.set_xlim(self.sim.x[0], self.sim.x[-1])
        self.ax.set_ylim(-0.05, 0.35)  # Set bounds cleanly above 0 to track the walls pointing up
        self.ax.set_xlabel('Position ($a_0$)')
        self.ax.set_ylabel('Probability Density / Potential ($a_0^{-1}$)')
        self.ax.set_title('Quantum Wave Packet Dynamics: Double Barrier Potential')
        self.ax.grid(True, linestyle='--', alpha=0.5)
        
        # Plot Upward Potential Walls (Scaled slightly to look optimal alongside probability line)
        self.ax.plot(
            self.sim.x,
            self.sim.potential * 0.15,
            color="red",
            lw=2,
            label="Potential Barrier"
        )
        
        # Plot Quantum Wave Line
        self.line, = self.ax.plot(self.sim.x, self.sim.prob_density, 
                                  color='indigo', lw=2, label=r'$|\Psi(x)|^2$')
        
        # Dynamic Diagnostic Overlay Data Text Box
        self.time_text = self.ax.text(0.02, 0.94, '', transform=self.ax.transAxes, 
                                      fontweight='bold', fontfamily='monospace',
                                      bbox=dict(facecolor='white', alpha=0.85))
        self.ax.legend(loc='upper right')

    def update(self, frame):
        """Standard non-generator update callback routing loop frames step by step."""
        self.sim.step()
        self.line.set_ydata(self.sim.prob_density)
        self.time += self.sim.dt
        
        # Standard conversion to standard physical femtoseconds
        current_fs_time = self.time * 0.0241888
        
        T = self.sim.transmission()
        R = self.sim.reflection()
        avg_x = self.sim.expectation_x()
        
        self.time_text.set_text(
            f"t    = {current_fs_time:6.2f} fs\n"
            f"T    = {T:6.3f}\n"
            f"R    = {R:6.3f}\n"
            f"<x>  = {avg_x:6.1f} a0"
        )
        return self.line, self.time_text

    def start(self):
        self.anim = animation.FuncAnimation(
            self.fig, 
            self.update, 
            frames=5000, 
            interval=15, 
            blit=False
        )
        plt.show()


if __name__ == "__main__":
    sim = QuantumSimulation(
        n_points=1600,             
        dt=0.4,                    
        barrier_height=1.5,        
        barrier_width=15.0,        
        central_barrier_width=6.0, 
        k0=2.0,                    
        x0=-150.0                  
    )
    
    animator = QuantumAnimator(sim)
    animator.start()
